In [6]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import json

from src.imagerie.ml_pipeline import build_feature_table, train_evaluate_ml
from src.imagerie.dl_pipeline import train_evaluate_dl
from src.imagerie.tfds_pipeline import load_tfds_pipeline

In [ ]:
OUTPUT_DIR = Path('outputs/notebook_run')
# DL variables
RUN_DL = False
DL_EPOCHS = 8
# limit ML dataset size to avoid memory overload
MAX_SAMPLES_ML = 1000 

In [ ]:
train_ds, info = load_tfds_pipeline(split="train")
class_names = info.features['label'].names
print(f'Total available classes: {len(class_names)}')

In [ ]:
# For ML we take a subset of data
ml_ds = train_ds.unbatch().take(MAX_SAMPLES_ML).batch(32)

X, y = build_feature_table(
    tfds_dataset=ml_ds,
    class_names=class_names,
    out_dir=OUTPUT_DIR,
)
print('Shape features:', X.shape)

In [ ]:
ml_summary = train_evaluate_ml(X, y, OUTPUT_DIR)
print(json.dumps(ml_summary, indent=2))

In [ ]:
summary = {'ml': ml_summary, 'classes': class_names, 'num_samples': int(len(X))}
if RUN_DL:
    dl_summary = train_evaluate_dl(
        tfds_dataset=train_ds, 
        class_names=class_names, 
        out_dir=OUTPUT_DIR / 'dl', 
        epochs=DL_EPOCHS
    )
    summary['dl'] = dl_summary
summary